In [1]:
import os
from pathlib import Path

import pickle
import json

import numpy as np
import pandas as pd

import mlflow
import mlflow.catboost

from catboost import CatBoostClassifier

import scipy.sparse as sp

from recsys_utils import evaluate_topk, load_clients_sample, log_ranker_run, print_metrics

# Настройки отображения
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# Пути проекта: данные в data/ (или в папке из переменной окружения DATA_DIR),
# модель и таблицы для сервиса сохраняются в services/
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = Path(os.getenv("DATA_DIR", PROJECT_ROOT / "data"))
DATA_PATH = DATA_DIR / "train_ver2.csv"
ASSETS_DIR = PROJECT_ROOT / "model_improvement" / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
SERVICE_DIR = PROJECT_ROOT / "services"

# MLflow: Tracking Server запускается скриптом services/run_mlflow.sh
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = "bank_products_recsys"
RUN_NAME = "stage4_experiments"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1792498042344, experiment_id='1', last_update_time=1792498042344, lifecycle_stage='active', name='bank_products_recsys', tags={}>

In [2]:
# Файл целиком не читается: берётся выборка клиентов (около 200 тысяч из 957 тысяч) с полной историей каждого
# и только те колонки, которые нужны для моделирования (остальные не используются, а занимают много памяти)
SAMPLE_PERCENT = 21

USER_COLS = [
    "ncodpers", "fecha_dato", "fecha_alta", "age", "antiguedad", "renta", "sexo", "ind_nuevo",
    "ind_actividad_cliente", "indfall", "indresi", "indext", "segmento", "tiprel_1mes", "canal_entrada",
]
all_cols = pd.read_csv(DATA_PATH, nrows=0).columns
use_cols = [c for c in all_cols if c in USER_COLS or (c.startswith("ind_") and c.endswith("_ult1"))]

df = load_clients_sample(DATA_PATH, SAMPLE_PERCENT, usecols=use_cols)

### Константы и первичные преобразования

In [3]:
ID_COL = "ncodpers"
DATE_COL = "fecha_dato"
JOIN_DATE_COL = "fecha_alta"

# Продуктовые колонки
PRODUCT_COLS = sorted([c for c in df.columns if c.startswith("ind_") and c.endswith("_ult1")])

print("Строк:", len(df))
print("Уникальных клиентов:", df[ID_COL].nunique())
print("Продуктовых колонок:", len(PRODUCT_COLS))

# Приведение типов и базовые колонки времени
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df[JOIN_DATE_COL] = pd.to_datetime(df[JOIN_DATE_COL], errors="coerce")

df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["antiguedad"] = pd.to_numeric(df["antiguedad"], errors="coerce")

df["month"] = df[DATE_COL].dt.to_period("M").dt.to_timestamp()
df["month_str"] = df["month"].dt.strftime("%Y-%m")

print("Диапазон дат:", df[DATE_COL].min(), " ", df[DATE_COL].max())
print("Уникальных месяцев:", df["month"].nunique())

# Удаление колонок с почти полной пустотой
HIGH_MISSING_COLS = ["conyuemp", "ult_fec_cli_1t"]
drop_cols = [c for c in HIGH_MISSING_COLS if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)
print("Удалены колонки:", drop_cols)

# Списки типов для дальнейшего использования
not_feature_cols = [ID_COL, DATE_COL, JOIN_DATE_COL, "month", "month_str"] + PRODUCT_COLS

NUMERIC_COLS = [
    c for c in df.select_dtypes(include=["int64", "float64"]).columns
    if c not in not_feature_cols
]

CATEGORICAL_COLS = [
    c for c in df.select_dtypes(include=["object", "category", "bool"]).columns
    if c not in not_feature_cols
]

print("Числовых признаков:", len(NUMERIC_COLS))
print("Категориальных признаков:", len(CATEGORICAL_COLS))


Строк: 2861344
Уникальных клиентов: 200645
Продуктовых колонок: 24


Диапазон дат: 2015-01-28 00:00:00   2016-05-28 00:00:00
Уникальных месяцев: 17
Удалены колонки: []


Числовых признаков: 5
Категориальных признаков: 7


### Уменьшение объема датафрейма

ядро всё время падает

In [4]:
import gc

# Выборка клиентов сделана при чтении файла (load_clients_sample), здесь только контроль размера
gc.collect()

print("Итог после сэмпла")
print("Строк:", len(df))
print("Пользователей:", df[ID_COL].nunique())
print("Уникальных месяцев:", df["month"].nunique())

Итог после сэмпла
Строк: 2861344
Пользователей: 200645
Уникальных месяцев: 17


# Построение baseline

# Разбиение данных

In [5]:
# Временное разбиение данных
SPLIT_DT = pd.to_datetime("2016-01-01")

df_train = df[df[DATE_COL] < SPLIT_DT].copy()
df_test = df[df[DATE_COL] >= SPLIT_DT].copy()

print("split_dt:", SPLIT_DT)

print("rows train:", len(df_train))
print("rows test :", len(df_test))

print("users train:", df_train[ID_COL].nunique())
print("users test :", df_test[ID_COL].nunique())

print("train date range:",
      df_train[DATE_COL].min(), df_train[DATE_COL].max())
print("test  date range:",
      df_test[DATE_COL].min(), df_test[DATE_COL].max())

print("train months:", df_train["month"].nunique())
print("test months :", df_test["month"].nunique())


split_dt: 2016-01-01 00:00:00
rows train: 1891836
rows test : 969508
users train: 194568
users test : 196835
train date range: 2015-01-28 00:00:00 2015-12-28 00:00:00
test  date range: 2016-01-28 00:00:00 2016-05-28 00:00:00
train months: 12
test months : 5


In [6]:
# Формирование событий user-product по train-периоду
events_train = (
    df_train[[ID_COL, "month"] + PRODUCT_COLS]
    .melt(
        id_vars=[ID_COL, "month"],
        var_name="product_id",
        value_name="value",
    )
)

# Оставляем только факты наличия продукта
events_train = events_train[events_train["value"] == 1].drop(columns=["value"])

print("events_train rows:", len(events_train))
print("users:", events_train[ID_COL].nunique())
print("products:", events_train["product_id"].nunique())
print("months:", events_train["month"].nunique())
print("date range:", events_train["month"].min(), events_train["month"].max())


events_train rows: 2896093
users: 148557
products: 24
months: 12


date range: 2015-01-01 00:00:00 2015-12-01 00:00:00


In [7]:
# Таргет: новые подключения продукта (0 -> 1)
LABEL_START_DT = pd.to_datetime("2015-12-01")
TARGET_START_DT = pd.to_datetime("2016-01-01")

df_label = df[df[DATE_COL] >= LABEL_START_DT].copy()
df_label = df_label.sort_values([ID_COL, "month"])

prev_products = df_label.groupby(ID_COL)[PRODUCT_COLS].shift(1)

new_products = (df_label[PRODUCT_COLS] == 1) & (prev_products == 0)

targets_pos = (
    new_products
    .assign(**{ID_COL: df_label[ID_COL].values, "month": df_label["month"].values})
    .melt(
        id_vars=[ID_COL, "month"],
        var_name="product_id",
        value_name="target",
    )
)

targets_pos = targets_pos[targets_pos["target"]].drop(columns=["target"])
targets_pos["target"] = 1
targets_pos = targets_pos[targets_pos["month"] >= TARGET_START_DT].copy()

print("positive targets:", len(targets_pos))
print("users:", targets_pos[ID_COL].nunique())
print("products:", targets_pos["product_id"].nunique())
print("months:", targets_pos["month"].nunique())


positive targets: 38825
users: 22664
products: 21
months: 5


In [8]:
# Кандидаты по популярности (top-N) и подготовка отрицательных примеров
# month уже рассчитан в df_test, повторно не считаем

print("Расчёт популярности продуктов по train-периоду")
product_popularity = (
    events_train["product_id"]
    .value_counts()
    .rename("cnt")
    .reset_index()
    .rename(columns={"product_id": "product_id"})
)

TOP_N = 10
top_products = product_popularity.head(TOP_N)["product_id"].tolist()
print("TOP_N:", TOP_N)
print("Топ продуктов для кандидатов:", top_products)

print("Формирование user-month для 2016")
users_months = df_test[[ID_COL, "month"]].drop_duplicates()
print("user-month строк:", len(users_months))
print("Пользователей:", users_months[ID_COL].nunique())
print("Месяцев:", users_months["month"].nunique())

print("Декартово произведение user-month x top_products")
top_df = pd.DataFrame({"product_id": top_products})
users_months["key"] = 1
top_df["key"] = 1

candidates = users_months.merge(top_df, on="key", how="inner").drop(columns=["key"])
users_months.drop(columns=["key"], inplace=True)
top_df.drop(columns=["key"], inplace=True)

print("Кандидатов всего:", len(candidates))

print("Разметка target по targets_pos")
pos_keys = targets_pos[[ID_COL, "month", "product_id"]].drop_duplicates()
pos_keys["target"] = 1

rank_df = candidates.merge(pos_keys, on=[ID_COL, "month", "product_id"], how="left")
rank_df["target"] = rank_df["target"].fillna(0).astype("int8")

print("Размер rank_df:", rank_df.shape)
print("Доля target=1:", round(float(rank_df["target"].mean()), 6))
print("Пользователей:", rank_df[ID_COL].nunique())
print("Месяцев:", rank_df["month"].nunique())
print("Продуктов:", rank_df["product_id"].nunique())


Расчёт популярности продуктов по train-периоду
TOP_N: 10
Топ продуктов для кандидатов: ['ind_cco_fin_ult1', 'ind_ctop_fin_ult1', 'ind_recibo_ult1', 'ind_ecue_fin_ult1', 'ind_cno_fin_ult1', 'ind_nom_pens_ult1', 'ind_nomina_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_dela_fin_ult1']
Формирование user-month для 2016
user-month строк: 969508
Пользователей: 196835


Месяцев: 5
Декартово произведение user-month x top_products


Кандидатов всего: 9695080
Разметка target по targets_pos


Размер rank_df: (9695080, 4)
Доля target=1: 0.003868
Пользователей: 196835
Месяцев: 5


Продуктов: 10


In [9]:
# TOP_K используется только для метрик и инференса
TOP_K = 5

print("Добавление score_pop и служебных признаков")

# score_pop считается один раз по product_popularity
score_pop = (
    product_popularity
    .assign(score_pop=lambda x: x["cnt"] / x["cnt"].max())
    .loc[lambda x: x["product_id"].isin(top_products), ["product_id", "score_pop"]]
)

# merge score_pop
rank_df = rank_df.merge(score_pop, on="product_id", how="left")
rank_df["score_pop"] = rank_df["score_pop"].fillna(0.0).astype("float32")

# категориальные признаки
rank_df["product_id"] = rank_df["product_id"].astype("category")
rank_df["month_str"] = rank_df["month"].dt.strftime("%Y-%m").astype("category")

print("Итоговая таблица для ранжирования")
print("Размер rank_df:", rank_df.shape)
print("Доля target=1:", round(float(rank_df["target"].mean()), 6))
print("Пользователей:", rank_df[ID_COL].nunique())
print("Месяцев:", rank_df["month"].nunique())
print("Продуктов:", rank_df["product_id"].nunique())


Добавление score_pop и служебных признаков


Итоговая таблица для ранжирования
Размер rank_df: (9695080, 6)
Доля target=1: 0.003868
Пользователей: 196835
Месяцев: 5
Продуктов: 10


In [10]:
print("Добавление позитивов вне top-N")

pos_all = targets_pos[[ID_COL, "month", "product_id"]].drop_duplicates()
pos_all["target"] = 1

# добавляем позитивы, которых нет в candidate set
rank_df = pd.concat(
    [rank_df, pos_all],
    ignore_index=True,
).drop_duplicates(subset=[ID_COL, "month", "product_id"], keep="first")

# пересчитать target по объединению
rank_df["target"] = rank_df["target"].fillna(0).astype("int8")

print("Размер rank_df после добавления позитивов:", rank_df.shape)
print("Продуктов:", rank_df["product_id"].nunique())
print("Доля target=1:", round(float(rank_df["target"].mean()), 6))


Добавление позитивов вне top-N


Размер rank_df после добавления позитивов: (9696400, 6)


Продуктов: 21
Доля target=1: 0.004004


In [11]:
# Сплит по времени внутри 2016: последний месяц берётся как valid
months_sorted = sorted(rank_df["month"].unique())
valid_month = months_sorted[-1]
rank_df["month_str"] = rank_df["month"].dt.strftime("%Y-%m")


train_df = rank_df[rank_df["month"] < valid_month].copy()
valid_df = rank_df[rank_df["month"] == valid_month].copy()

print("Месяц валидации:", valid_month)
print("train:", train_df.shape, "valid:", valid_df.shape)
print("Доля target=1 train:", round(float(train_df["target"].mean()), 6))
print("Доля target=1 valid:", round(float(valid_df["target"].mean()), 6))

# Освобождаем память: промежуточные таблицы дальше не нужны, а обучение CatBoost требует много ОЗУ
import gc

del df_train, df_test, df_label, prev_products, new_products, candidates, users_months, top_df, pos_keys, pos_all, rank_df
gc.collect()

Месяц валидации: 2016-05-01 00:00:00
train: (7742443, 6) valid: (1953957, 6)
Доля target=1 train: 0.004021
Доля target=1 valid: 0.003939


57

In [12]:
FEATURES = ["score_pop", "month_str", "product_id"]
CAT_FEATURES = ["month_str", "product_id"]

X_train = train_df[FEATURES].copy()
y_train = train_df["target"].astype("int8").copy()

X_valid = valid_df[FEATURES].copy()
y_valid = valid_df["target"].astype("int8").copy()

model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
    # признаки с малым числом значений (продукт, месяц) кодируются one-hot: так CatBoost не считает счётчики категорий
    # (CTR) и тратит заметно меньше памяти
    one_hot_max_size=32,
)

print("Обучение CatBoost")
model.fit(
    X_train, y_train,
    cat_features=CAT_FEATURES,
    eval_set=(X_valid, y_valid),
)

Обучение CatBoost


0:	learn: 0.4543470	test: 0.4508638	best: 0.4508638 (0)	total: 754ms	remaining: 3m 45s


50:	learn: 0.0237585	test: 0.0237507	best: 0.0237503 (45)	total: 26.8s	remaining: 2m 10s


100:	learn: 0.0237390	test: 0.0237507	best: 0.0237496 (77)	total: 51.5s	remaining: 1m 41s


150:	learn: 0.0237375	test: 0.0237494	best: 0.0237490 (110)	total: 1m 15s	remaining: 1m 14s


200:	learn: 0.0237371	test: 0.0237497	best: 0.0237490 (110)	total: 1m 39s	remaining: 49.2s


250:	learn: 0.0237366	test: 0.0237496	best: 0.0237490 (110)	total: 2m 3s	remaining: 24.2s


299:	learn: 0.0237366	test: 0.0237496	best: 0.0237490 (110)	total: 2m 26s	remaining: 0us

bestTest = 0.02374899923
bestIteration = 110

Shrink model to first 111 iterations.


In [13]:
print("Скоринг валидации")
valid_scored = valid_df[[ID_COL, "month", "product_id", "target"]].copy()
valid_scored["pred"] = model.predict_proba(X_valid)[:, 1].astype("float32")

metrics = evaluate_topk(valid_scored, TOP_K, ID_COL)

print_metrics(metrics)

Скоринг валидации


Precision@k (all): 0.006587
Recall@k    (all): 0.025193
MAP@k       (all): 0.016498
NDCG@k      (all): 0.018913
Precision@k (pos): 0.216448
Recall@k    (pos): 0.827809
MAP@k       (pos): 0.542105
NDCG@k      (pos): 0.621452
Coverage users: 1.0
Coverage items: 0.736842


In [14]:
metadata = {
    "stage": "baseline_ranking",
    "candidate_generator": "popularity_topN",
    "task_type": "product_recommendation_ranking",
    "time_split": "train: 2015, label: 2016, valid: 2016-05",
    "features": FEATURES,
    "cat_features": CAT_FEATURES,
    "top_k": TOP_K,
    "top_n": TOP_N,
}

run_id, model_uri = log_ranker_run(
    run_name="baseline_popularity_catboost",
    ranker=model,
    X_valid=X_valid,
    metrics=metrics,
    params={"top_k": TOP_K, "top_n": TOP_N, "top_products_n": len(top_products)},
    metadata=metadata,
)

run_id, model_uri

2026/10/20 12:48:47 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



('2d7ae4cb39284a1ebf43836b4dabde3d',
 'runs:/2d7ae4cb39284a1ebf43836b4dabde3d/model')

Оценка качества модели

Оценка проведена на валидационном периоде: последний месяц 2016 года.  
Модель обучалась как ранкер с кандидатами, сформированными по популярности продуктов (top-N).

Используемые метрики

Метрики рассчитывались для рекомендаций top-K продуктов в каждом `(user, month)`.

- **Precision@K (all)** — средняя доля релевантных рекомендаций среди всех user-month
- **Recall@K (all)** — средняя доля найденных релевантных продуктов среди всех user-month
- **Precision@K (pos)** — precision только для user-month, где был хотя бы один новый продукт
- **Recall@K (pos)** — recall только для user-month, где был хотя бы один новый продукт
- **MAP@K** — средняя точность с учётом позиции верных продуктов в списке (учитывает порядок)
- **NDCG@K** — качество ранжирования с учётом позиции: верный продукт в начале списка даёт больший вклад
- **Coverage users** — доля user-month, для которых выданы рекомендации
- **Coverage items** — доля продуктов, которые хотя бы раз попали в рекомендации

Интерпретация результатов

- Низкие значения **Precision@K (all)** и **Recall@K (all)** являются ожидаемыми, так как большинство user-month не содержат новых подключений, при этом рекомендации выдаются всегда.
- Метрики **Precision@K (pos)**, **Recall@K (pos)**, **MAP@K (pos)** и **NDCG@K (pos)** отражают качество ранжирования в релевантных случаях и являются основными для данной постановки задачи.
- Высокий **Recall@K (pos)** показывает, что модель в большинстве случаев включает реально подключённый продукт в top-K рекомендаций.
- **Coverage users = 1.0** означает, что рекомендации сформированы для всех user-month.
- **Coverage items ≈ 0.74** указывает, что модель использует большую часть доступных продуктов в рекомендациях, без концентрации на одном-двух популярных.


Вывод

Модель демонстрирует корректное качество ранжирования для базовой версии без использования сложных признаков.  
Результаты могут служить отправной точкой для дальнейших улучшений модели.


# ALS

попробуем использовать ALS

In [15]:
# Кодирование пользователей и продуктов для ALS
# Используется только train-период, без утечек из 2016

user_codes, user_index = pd.factorize(
    events_train[ID_COL],
    sort=True
)
events_train["user_id_enc"] = user_codes.astype("int32")

item_codes, item_index = pd.factorize(
    events_train["product_id"],
    sort=True
)
events_train["item_id_enc"] = item_codes.astype("int16")

n_users = events_train["user_id_enc"].nunique()
n_items = events_train["item_id_enc"].nunique()

print("Число пользователей:", n_users)
print("Число продуктов:", n_items)
print("Ожидаемое число продуктов:", len(PRODUCT_COLS))


Число пользователей: 148557
Число продуктов: 24
Ожидаемое число продуктов: 24


In [16]:
# Построение user-item матрицы для ALS
rows = events_train["user_id_enc"].to_numpy()
cols = events_train["item_id_enc"].to_numpy()

user_item_matrix = sp.csr_matrix(
    (np.ones(len(events_train), dtype=np.float32), (rows, cols)),
    shape=(n_users, n_items),
)

print("Размер матрицы:", user_item_matrix.shape)
print("Ненулевых:", user_item_matrix.nnz)

# item-user матрица готовится только при необходимости, для текущего кода не нужна
# item_user_matrix = user_item_matrix.T.tocsr()
# print("Размер item-user:", item_user_matrix.shape)

from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(
    factors=32,
    iterations=20,
    regularization=0.05,
    random_state=42,
)

print("Обучение ALS")
als_model.fit(user_item_matrix)

print("user_factors:", als_model.user_factors.shape)
print("item_factors:", als_model.item_factors.shape)


/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Размер матрицы: (148557, 24)
Ненулевых: 293985
Обучение ALS


  0%|          | 0/20 [00:00<?, ?it/s]

  5%|▌         | 1/20 [00:01<00:24,  1.27s/it]

 10%|█         | 2/20 [00:02<00:20,  1.16s/it]

...

 90%|█████████ | 18/20 [00:20<00:02,  1.19s/it]

 95%|█████████▌| 19/20 [00:21<00:01,  1.17s/it]

100%|██████████| 20/20 [00:22<00:00,  1.21s/it]

100%|██████████| 20/20 [00:22<00:00,  1.15s/it]

user_factors: (148557, 32)
item_factors: (24, 32)


In [17]:
# Пользователи 2016, которые присутствуют в train (ALS может рекомендовать только им)
valid_users = targets_pos[[ID_COL]].drop_duplicates()
valid_users = valid_users[valid_users[ID_COL].isin(user_index)].copy()

print("Пользователей для рекомендаций:", len(valid_users))

# Маппинг user_id -> user_id_enc
user_to_enc = pd.Series(np.arange(len(user_index), dtype="int32"), index=user_index)
valid_users["user_id_enc"] = valid_users[ID_COL].map(user_to_enc).astype("int32")

TOP_K_ALS = 10
user_ids_enc = valid_users["user_id_enc"].to_numpy()

print("Генерация ALS кандидатов, TOP_K_ALS:", TOP_K_ALS)

item_ids_enc, scores = als_model.recommend(
    user_ids_enc,
    user_item_matrix[user_ids_enc],
    N=TOP_K_ALS,
    filter_already_liked_items=True,
)


Пользователей для рекомендаций: 19858
Генерация ALS кандидатов, TOP_K_ALS: 10


In [18]:
print("Сбор ALS кандидатов в таблицу")

als_candidates = pd.DataFrame(
    {
        "user_id_enc": np.repeat(user_ids_enc, TOP_K_ALS).astype("int32"),
        "item_id_enc": np.asarray(item_ids_enc, dtype="int16").ravel(),
        "score_als": np.asarray(scores, dtype="float32").ravel(),
    }
)

# Декодирование id обратно в исходные значения
enc_to_user = pd.Series(user_index, index=np.arange(len(user_index), dtype="int32"))
enc_to_item = pd.Series(item_index, index=np.arange(len(item_index), dtype="int16"))

als_candidates[ID_COL] = als_candidates["user_id_enc"].map(enc_to_user)
als_candidates["product_id"] = als_candidates["item_id_enc"].map(enc_to_item)

als_candidates = als_candidates[[ID_COL, "product_id", "score_als"]]
als_candidates = als_candidates.sort_values([ID_COL, "score_als"], ascending=[True, False])

print("Строк кандидатов ALS:", len(als_candidates))
print("Пользователей:", als_candidates[ID_COL].nunique())
print("Продуктов:", als_candidates["product_id"].nunique())

print("Добавление месяцев 2016 к ALS-кандидатам")
months_2016 = sorted(targets_pos["month"].unique())
print("Месяцев 2016:", len(months_2016))
print("Диапазон:", months_2016[0], months_2016[-1])

als_candidates["key"] = 1
months_df = pd.DataFrame({"month": months_2016})
months_df["key"] = 1

als_candidates_month = als_candidates.merge(months_df, on="key", how="inner").drop(columns=["key"])
als_candidates.drop(columns=["key"], inplace=True)
months_df.drop(columns=["key"], inplace=True)

als_candidates_month["month_str"] = als_candidates_month["month"].dt.strftime("%Y-%m").astype("category")
als_candidates_month["product_id"] = als_candidates_month["product_id"].astype("category")

print("ALS кандидаты с месяцами:", als_candidates_month.shape)


Сбор ALS кандидатов в таблицу
Строк кандидатов ALS: 198580
Пользователей: 19858
Продуктов: 24
Добавление месяцев 2016 к ALS-кандидатам
Месяцев 2016: 5
Диапазон: 2016-01-01 00:00:00 2016-05-01 00:00:00


ALS кандидаты с месяцами: (992900, 5)


In [19]:
print("Разметка ALS кандидатов таргетом 0->1")

pos = targets_pos[[ID_COL, "month", "product_id"]].drop_duplicates().copy()
pos["target"] = 1

rank_als = als_candidates_month.merge(
    pos,
    on=[ID_COL, "month", "product_id"],
    how="left",
)

rank_als["target"] = rank_als["target"].fillna(0).astype("int8")

print("Размер таблицы ранжирования ALS:", rank_als.shape)
print("Доля target=1:", round(float(rank_als["target"].mean()), 6))
print("Пользователей:", rank_als[ID_COL].nunique())
print("Месяцев:", rank_als["month"].nunique())
print("Продуктов:", rank_als["product_id"].nunique())


Разметка ALS кандидатов таргетом 0->1


Размер таблицы ранжирования ALS: (992900, 6)
Доля target=1: 0.007523
Пользователей: 19858
Месяцев: 5
Продуктов: 24


In [20]:
TOP_K = 5

print("Подготовка таблицы ранжирования ALS")

rank_als["score_als"] = rank_als["score_als"].astype("float32")
rank_als["target"] = rank_als["target"].astype("int8")
rank_als["product_id"] = rank_als["product_id"].astype("category")
rank_als["month_str"] = rank_als["month"].dt.strftime("%Y-%m").astype("category")

FEATURES = ["score_als", "month_str", "product_id"]
CAT_FEATURES = ["month_str", "product_id"]

# Сплит по времени внутри 2016: последний месяц = valid
months_sorted = sorted(rank_als["month"].unique())
valid_month = months_sorted[-1]

train_df = rank_als[rank_als["month"] < valid_month].copy()
valid_df = rank_als[rank_als["month"] == valid_month].copy()

print("Месяц валидации:", valid_month)
print("train:", train_df.shape, "valid:", valid_df.shape)
print("Доля target=1 train:", round(float(train_df["target"].mean()), 6))
print("Доля target=1 valid:", round(float(valid_df["target"].mean()), 6))


Подготовка таблицы ранжирования ALS


Месяц валидации: 2016-05-01 00:00:00
train: (794320, 6) valid: (198580, 6)
Доля target=1 train: 0.007628
Доля target=1 valid: 0.007105


In [21]:
print("Обучение CatBoost ранкера на ALS кандидатах")

X_train = train_df[FEATURES].copy()
y_train = train_df["target"].astype("int8").copy()

X_valid = valid_df[FEATURES].copy()
y_valid = valid_df["target"].astype("int8").copy()

ranker_als = CatBoostClassifier(
    iterations=400,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
    # признаки с малым числом значений (продукт, месяц) кодируются one-hot: так CatBoost не считает счётчики категорий
    # (CTR) и тратит заметно меньше памяти
    one_hot_max_size=32,
    early_stopping_rounds=50,
)

ranker_als.fit(
    X_train, y_train,
    cat_features=CAT_FEATURES,
    eval_set=(X_valid, y_valid),
)

print("Скоринг валидации")
valid_scored = valid_df[[ID_COL, "month", "product_id", "target"]].copy()
valid_scored["pred"] = ranker_als.predict_proba(X_valid)[:, 1].astype("float32")

metrics = evaluate_topk(valid_scored, TOP_K, ID_COL)

print_metrics(metrics)

print("Расчёт novelty/diversity")
user_seen = events_train.groupby(ID_COL)["product_id"].apply(set)

pred_sets = metrics["pred_sets"]

novelties = []
for key, recs in pred_sets.items():
    user_id = key[0]
    seen = user_seen.get(user_id, set())
    unknown = sum(1 for p in recs if p not in seen)
    novelties.append(unknown / len(recs))

novelty_at_k = float(np.mean(novelties)) if novelties else 0.0

diversities = []
for recs in pred_sets.values:
    diversities.append(len(set(recs)) / len(recs))

diversity_at_k = float(np.mean(diversities)) if diversities else 0.0

print("Novelty@k:", round(novelty_at_k, 6))
print("Diversity@k:", round(diversity_at_k, 6))

Обучение CatBoost ранкера на ALS кандидатах


0:	learn: 0.4692472	test: 0.4699897	best: 0.4699897 (0)	total: 79.1ms	remaining: 31.6s


50:	learn: 0.0336225	test: 0.0325140	best: 0.0325140 (50)	total: 3.19s	remaining: 21.8s


100:	learn: 0.0324787	test: 0.0317384	best: 0.0317270 (98)	total: 6.43s	remaining: 19.1s


150:	learn: 0.0320491	test: 0.0315062	best: 0.0315060 (146)	total: 9.7s	remaining: 16s


...

300:	learn: 0.0315578	test: 0.0313504	best: 0.0313504 (300)	total: 19.7s	remaining: 6.47s


350:	learn: 0.0314478	test: 0.0313385	best: 0.0313369 (347)	total: 23s	remaining: 3.21s


399:	learn: 0.0313421	test: 0.0312934	best: 0.0312906 (398)	total: 26.4s	remaining: 0us

bestTest = 0.0312906265
bestIteration = 398

Shrink model to first 399 iterations.
Скоринг валидации


Precision@k (all): 0.013758
Recall@k    (all): 0.049753
MAP@k       (all): 0.038128
NDCG@k      (all): 0.041144
Precision@k (pos): 0.264729
Recall@k    (pos): 0.957364
MAP@k       (pos): 0.733659
NDCG@k      (pos): 0.791712
Coverage users: 1.0
Coverage items: 0.958333
Расчёт novelty/diversity


Novelty@k: 1.0
Diversity@k: 1.0


In [22]:
RUN_NAME = "als_candidates_catboost_ranker"
REGISTRY_MODEL_NAME = "bank_products_ranker_als"

metadata = {
    "stage": "ranking_with_als_candidates",
    "candidate_generator": "als",
    "task_type": "product_recommendation_ranking",
    "time_split": "train: 2015, label: 2016, valid: 2016-05",
    "features": FEATURES,
    "cat_features": CAT_FEATURES,
    "top_k": TOP_K,
    "top_k_als": TOP_K_ALS,
}

run_id, model_uri = log_ranker_run(
    run_name=RUN_NAME,
    ranker=ranker_als,
    X_valid=X_valid,
    metrics=metrics,
    params={"top_k": TOP_K, "top_k_als": TOP_K_ALS},
    extra_metrics={"novelty_at_k": novelty_at_k, "diversity_at_k": diversity_at_k},
    metadata=metadata,
    registered_model_name=REGISTRY_MODEL_NAME,
)

run_id, model_uri

/usr/local/lib/python3.10/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


Successfully registered model 'bank_products_ranker_als'.
2026/10/20 12:50:04 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: bank_products_ranker_als, version 1


Created version '1' of model 'bank_products_ranker_als'.


('5ea585a49d9e432789634f832b947a31',
 'runs:/5ea585a49d9e432789634f832b947a31/model')

Сравнительный вывод по моделям

Оценка проведена на одном и том же валидационном периоде: последний месяц 2016 года.  
Во всех экспериментах использовалась одинаковая схема оценки top-K рекомендаций.

Сводка метрик

| Модель                              | Precision@K (all) | Recall@K (all) | MAP@K (all) | NDCG@K (all) | Precision@K (pos) | Recall@K (pos) | MAP@K (pos) | NDCG@K (pos) | Coverage items |
|------------------------------------|-------------------|----------------|-------------|--------------|-------------------|----------------|-------------|--------------|----------------|
| Popularity + CatBoost (baseline)   | 0.0066            | 0.0252         | 0.0165      | 0.0189       | 0.216             | 0.828          | 0.542       | 0.621        | 0.74           |
| ALS + CatBoost (ранкер)            | 0.0138            | 0.0498         | 0.0381      | 0.0411       | 0.265             | 0.957          | 0.734       | 0.792        | 0.96           |

Охват клиентов (Coverage users) равен 1.0 для обеих моделей.

Novelty@K = 1.0  
Diversity@K = 1.0  
(для обеих моделей, что является следствием фильтрации уже имеющихся продуктов и малого каталога)

Интерпретация

- **ALS + CatBoost стабильно превосходит popularity-baseline** по всем ключевым метрикам.
- Рост **Precision@K (all)** и **Recall@K (all)** означает, что модель лучше ранжирует кандидатов даже в общей массе user-month, где большинство случаев не содержит покупок.
- Существенный рост **Recall@K (pos)** (≈0.96 против ≈0.83) показывает, что ALS-кандидаты почти всегда содержат реально подключённый продукт в top-K.
- Рост **MAP@K** и **NDCG@K** показывает, что верные продукты оказываются выше в списке, а не только попадают в него.
- Увеличение **Precision@K (pos)** подтверждает, что улучшение recall не достигается за счёт деградации точности.
- **Coverage items ≈ 0.96** у ALS-подхода означает, что модель использует весь каталог продуктов, а не концентрируется только на топ-популярных.
- Значения **Novelty@K** и **Diversity@K** не являются информативными для сравнения моделей в данной задаче и интерпретируются как свойство постановки задачи, а не качества алгоритма.

Итог

Использование ALS для кандидатогенерации в сочетании с табличным ранжированием на CatBoost даёт заметный прирост качества по сравнению с popularity-baseline.  
Коллаборативный сигнал в данной постановке **работает и полезен**, несмотря на небольшой размер каталога.



# Добавление остальных признаков + генерация новых

In [23]:
rank_als

,ncodpers,product_id,score_als,month,month_str,target
0,15889,ind_fond_fin_ult1,0.003745,2016-01-01,2016-01,0
1,15889,ind_fond_fin_ult1,0.003745,2016-02-01,2016-02,0
2,15889,ind_fond_fin_ult1,0.003745,2016-03-01,2016-03,0
3,15889,ind_fond_fin_ult1,0.003745,2016-04-01,2016-04,0
4,15889,ind_fond_fin_ult1,0.003745,2016-05-01,2016-05,0
...,...,...,...,...,...,...
992895,1520050,ind_dela_fin_ult1,0.000275,2016-01-01,2016-01,0
992896,1520050,ind_dela_fin_ult1,0.000275,2016-02-01,2016-02,0
992897,1520050,ind_dela_fin_ult1,0.000275,2016-03-01,2016-03,0
992898,1520050,ind_dela_fin_ult1,0.000275,2016-04-01,2016-04,0


In [24]:
# ====== user_features: демография + история (t-1), без тяжелого rolling/apply

need_cols = [
    ID_COL, DATE_COL,
    "age", "renta", "antiguedad",
    "sexo", "ind_nuevo", "ind_actividad_cliente",
    "indfall", "indresi", "indext",
    "segmento", "tiprel_1mes", "canal_entrada", "month"
] + PRODUCT_COLS

base = df[need_cols].copy()

base = base.sort_values([ID_COL, "month"])

# числовые
base["renta"] = pd.to_numeric(base["renta"], errors="coerce")
base["log_renta"] = np.log1p(base["renta"])

base["month"] = pd.to_datetime(base["month"], errors="coerce")


# продуктовая матрица
prod = base[PRODUCT_COLS].fillna(0).astype("int8")
prev_prod = prod.groupby(base[ID_COL]).shift(1).fillna(0).astype("int8")

# n_products_prev
base["n_products_prev"] = prev_prod.sum(axis=1).astype("int16")

# есть ли новое подключение в текущем месяце (0->1)
has_new_now = ((prod == 1) & (prev_prod == 0)).any(axis=1).astype("int8")
base["has_new_prev"] = has_new_now.astype("int8")

# новые подключения за последние 3 месяца (по флагу has_new_now)
# это не "сколько продуктов", а "сколько месяцев с новым продуктом" за окно
base["new_cnt_last_3m"] = (
    has_new_now
    .groupby(base[ID_COL])
    .rolling(3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
    .astype("int8")
)

# months_since_last_new (по месяцам, без apply)
# last_new_month = последний месяц, когда has_new_now == 1
base["_last_new_month"] = base["month"].where(has_new_now == 1)
base["_last_new_month"] = base.groupby(ID_COL)["_last_new_month"].ffill()

base["months_since_last_new"] = (
    ((base["month"] - base["_last_new_month"]).dt.days // 30)
    .fillna(99)
    .astype("int16")
)

base = base.drop(columns=["_last_new_month"])

user_features = base[
    [
        ID_COL, "month",
        "age", "log_renta", "antiguedad",
        "sexo", "ind_nuevo", "ind_actividad_cliente",
        "indfall", "indresi", "indext",
        "segmento", "tiprel_1mes", "canal_entrada",
        "n_products_prev", "has_new_prev",
        "new_cnt_last_3m", "months_since_last_new",
    ]
].copy()

# ====== merge
user_features["month_str"] = user_features["month"].dt.strftime("%Y-%m")

rank_full = rank_als.merge(
    user_features.drop(columns=["month"]),
    on=[ID_COL, "month_str"],
    how="left"
)

CAT_FEATURES = [
    "month_str", "product_id",
    "sexo", "ind_nuevo", "ind_actividad_cliente",
    "indfall", "indresi", "indext",
    "segmento", "tiprel_1mes", "canal_entrada",
]

# NaN в cat: сразу в строку, чтобы CatBoost не падал
for c in CAT_FEATURES:
    rank_full[c] = rank_full[c].astype("object").fillna("unknown").astype(str)

NUM_FEATURES = [
    "score_als",
    "age", "log_renta", "antiguedad",
    "n_products_prev", "has_new_prev",
    "new_cnt_last_3m", "months_since_last_new",
]

FEATURES = NUM_FEATURES + CAT_FEATURES

# split
months_sorted = sorted(rank_full["month_str"].unique())
valid_month_str = months_sorted[-1]

train_df = rank_full[rank_full["month_str"] < valid_month_str]
valid_df = rank_full[rank_full["month_str"] == valid_month_str]

print("Месяц валидации:", valid_month_str)
print("train:", train_df.shape, "valid:", valid_df.shape)


X_train = train_df[FEATURES]
y_train = train_df["target"].astype("int8")

X_valid = valid_df[FEATURES]
y_valid = valid_df["target"].astype("int8")

print("Таблица ранжирования с фичами")
print("train:", X_train.shape, "valid:", X_valid.shape)

# ====== обучение
ranker = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
    # признаки с малым числом значений (продукт, месяц) кодируются one-hot: так CatBoost не считает счётчики категорий
    # (CTR) и тратит заметно меньше памяти
    one_hot_max_size=32,
)

print("Обучение CatBoost")
ranker.fit(
    X_train, y_train,
    cat_features=CAT_FEATURES,
    eval_set=(X_valid, y_valid),
)

Месяц валидации: 2016-05
train: (794320, 22) valid: (198580, 22)
Таблица ранжирования с фичами
train: (794320, 19) valid: (198580, 19)
Обучение CatBoost


0:	learn: 0.4518729	test: 0.4847254	best: 0.4847254 (0)	total: 323ms	remaining: 1m 36s


50:	learn: 0.0216474	test: 0.0201818	best: 0.0201818 (50)	total: 13.3s	remaining: 1m 4s


100:	learn: 0.0200609	test: 0.0187656	best: 0.0187656 (100)	total: 26.3s	remaining: 51.9s


150:	learn: 0.0192487	test: 0.0182878	best: 0.0182878 (150)	total: 39.4s	remaining: 38.9s


200:	learn: 0.0187086	test: 0.0180358	best: 0.0180358 (200)	total: 52.2s	remaining: 25.7s


250:	learn: 0.0183142	test: 0.0178623	best: 0.0178604 (249)	total: 1m 5s	remaining: 12.7s


299:	learn: 0.0180296	test: 0.0177747	best: 0.0177747 (299)	total: 1m 17s	remaining: 0us

bestTest = 0.01777466038
bestIteration = 299



In [25]:
valid_df["month"] = pd.to_datetime(valid_df["month_str"], format="%Y-%m")
valid_scored = valid_df[[ID_COL, "month", "product_id", "target"]].copy()

valid_scored["pred"] = ranker.predict_proba(X_valid)[:, 1].astype("float32")

metrics = evaluate_topk(valid_scored, TOP_K, ID_COL)

print_metrics(metrics)



print("Расчёт novelty/diversity")
user_seen = events_train.groupby(ID_COL)["product_id"].apply(set)

pred_sets = metrics["pred_sets"]

novelties = []
for key, recs in pred_sets.items():
    user_id = key[0]
    seen = user_seen.get(user_id, set())
    unknown = sum(1 for p in recs if p not in seen)
    novelties.append(unknown / len(recs))

novelty_at_k = float(np.mean(novelties)) if novelties else 0.0

diversities = []
for recs in pred_sets.values:
    diversities.append(len(set(recs)) / len(recs))

diversity_at_k = float(np.mean(diversities)) if diversities else 0.0

print("Novelty@k:", round(novelty_at_k, 6))
print("Diversity@k:", round(diversity_at_k, 6))

/tmp/ipykernel_30/1316730290.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_df["month"] = pd.to_datetime(valid_df["month_str"], format="%Y-%m")


Precision@k (all): 0.013748
Recall@k    (all): 0.04977
MAP@k       (all): 0.040543
NDCG@k      (all): 0.043019
Precision@k (pos): 0.264535
Recall@k    (pos): 0.957687
MAP@k       (pos): 0.780144
NDCG@k      (pos): 0.827775
Coverage users: 1.0
Coverage items: 1.0
Расчёт novelty/diversity


Novelty@k: 1.0
Diversity@k: 1.0


Интерпретация результатов

1. Precision@k (all) остаётся низкой (~1.4%)
Это ожидаемо и корректно для задачи с очень редким таргетом (≈0.4–0.7%) и большим числом отрицательных user–product пар
Метрика отражает общий шум задачи и не является основной для бизнес-оценки.

2. Precision@k (pos) ≈ 26%
Для user–month, где действительно произошло хотя бы одно новое подключение, где каждый четвёртый рекомендованный продукт оказывается релевантным

3. Recall@k (pos) ≈ 95.8%
Модель почти всегда попадает хотя бы в один реальный новый продукт клиента.
Это ключевая метрика для сценариев кросс-селла и апсейла.

4. Coverage users = 1.0
Рекомендации формируются для всех user–month в валидации.
Модель полностью покрывает целевую аудиторию.

5. Coverage items = 1.0
Модель использует весь каталог продуктов, т.е.отсутствие вырождения в популярностную модель.

6. Novelty@k = 1.0
Все рекомендованные продукты отсутствовали у пользователя ранее, то есть модель не рекомендует уже подключённые продукты.

7. MAP@k (pos) ≈ 0.78 и NDCG@k (pos) ≈ 0.83
Верные продукты в среднем стоят в начале списка: учёт позиции подтверждает качество ранжирования.

8. Diversity@k = 1.0
В каждом топ-k:

- все продукты уникальны
- ранжирование не схлопывается в повтор одного и того же продукта.

Сравнение с предыдущими этапами

Добавление демографических признаков и истории пользователя:

- не ухудшило recall
- сохранило precision@k (pos)
- не привело к росту popularity bias

Модель осталась стабильной, предсказуемой и интерпретируемой.

In [26]:
RUN_NAME = "als_ranker_with_demo_history"
REGISTRY_MODEL_NAME = "bank_products_ranker_als"

art_dir = "artifacts_tmp"
os.makedirs(art_dir, exist_ok=True)

# ALS факторы
np.save(os.path.join(art_dir, "als_user_factors.npy"), als_model.user_factors.astype("float32"))
np.save(os.path.join(art_dir, "als_item_factors.npy"), als_model.item_factors.astype("float32"))

# маппинги
# user_to_enc: Series index=ncodpers, value=user_id_enc
with open(os.path.join(art_dir, "user_to_enc.pkl"), "wb") as f:
    pickle.dump(user_to_enc, f)

# item_index: список product_id по item_id_enc
with open(os.path.join(art_dir, "enc_to_item.pkl"), "wb") as f:
    pickle.dump(enc_to_item, f)

# обратный маппинг product_id -> item_id_enc 
item_to_enc = {v: int(k) for k, v in enc_to_item.items()}
with open(os.path.join(art_dir, "item_to_enc.pkl"), "wb") as f:
    pickle.dump(item_to_enc, f)

# конфиг инференса
infer_cfg = {
    "top_k": int(TOP_K),
    "top_k_als": int(TOP_K_ALS),
    "features": FEATURES,
    "cat_features": CAT_FEATURES,
    "numeric_features": NUM_FEATURES,
    "product_cols": PRODUCT_COLS,
}
with open(os.path.join(art_dir, "inference_config.json"), "w", encoding="utf-8") as f:
    json.dump(infer_cfg, f, ensure_ascii=False, indent=2)

# log в MLflow
metadata = {
    "stage": "ranking_with_als_candidates_",
    "candidate_generator": "als",
    "task_type": "product_recommendation_ranking",
    "time_split": "train: 2015, label: 2016, valid: last month of 2016",
    "top_k": int(TOP_K),
    "top_k_als": int(TOP_K_ALS),
}

run_id, model_uri = log_ranker_run(
    run_name=RUN_NAME,
    ranker=ranker,
    X_valid=X_valid,
    metrics=metrics,
    params={"model_type": "CatBoostClassifier"},
    extra_metrics={"novelty_at_k": novelty_at_k, "diversity_at_k": diversity_at_k},
    metadata=metadata,
    registered_model_name=REGISTRY_MODEL_NAME,
)

print("run_id:", run_id)
print("model_uri:", model_uri)

/usr/local/lib/python3.10/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


Registered model 'bank_products_ranker_als' already exists. Creating a new version of this model...
2026/10/20 12:51:59 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: bank_products_ranker_als, version 2


run_id: c8a6ffa865bf49a2ab3504eb11fd672e
model_uri: runs:/c8a6ffa865bf49a2ab3504eb11fd672e/model


Created version '2' of model 'bank_products_ranker_als'.


# Сохраняем данные для прода

In [27]:
# Модель и таблицы для сервиса сохраняются рядом с его кодом: services/models и services/assets
MODELS_DIR = SERVICE_DIR / "models"
SERVICE_ASSETS_DIR = SERVICE_DIR / "assets"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SERVICE_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

ranker.save_model(str(MODELS_DIR / "model.cbm"))
print("saved model:", MODELS_DIR / "model.cbm")

TOP_K_ALS = 10

personal_als = als_candidates_month.copy()
personal_als = personal_als.sort_values(
    [ID_COL, "month_str", "score_als"],
    ascending=[True, True, False]
)
personal_als["rank_als"] = personal_als.groupby([ID_COL, "month_str"]).cumcount() + 1
personal_als = personal_als[personal_als["rank_als"] <= TOP_K_ALS].copy()

personal_als = personal_als[[ID_COL, "month_str", "product_id", "score_als", "rank_als"]]
personal_als["score_als"] = personal_als["score_als"].astype("float32")
personal_als["rank_als"] = personal_als["rank_als"].astype("int16")

print("personal_als:", personal_als.shape)

personal_als.to_parquet(SERVICE_ASSETS_DIR / "personal_als.parquet", index=False)
print("saved:", SERVICE_ASSETS_DIR / "personal_als.parquet")


user_feat_save = user_features.copy()
user_feat_save["month_str"] = user_feat_save["month"].dt.strftime("%Y-%m")
user_feat_save = user_feat_save.drop(columns=["month"], errors="ignore")

print("user_features:", user_feat_save.shape)

user_feat_save.to_parquet(SERVICE_ASSETS_DIR / "user_features.parquet", index=False)
print("saved:", SERVICE_ASSETS_DIR / "user_features.parquet")

infer_cfg = {
    "top_k": int(TOP_K),
    "top_k_als": int(TOP_K_ALS),
    "features": FEATURES,
    "cat_features": CAT_FEATURES,
    "numeric_features": NUM_FEATURES,
    "product_cols": PRODUCT_COLS,
}

with open(SERVICE_ASSETS_DIR / "inference_config.json", "w", encoding="utf-8") as f:
    json.dump(infer_cfg, f, ensure_ascii=False, indent=2)

print("saved:", SERVICE_ASSETS_DIR / "inference_config.json")

saved model: /app/services/models/model.cbm


personal_als: (992900, 5)


saved: /app/services/assets/personal_als.parquet


user_features: (2861344, 18)


saved: /app/services/assets/user_features.parquet
saved: /app/services/assets/inference_config.json
